<a href="https://colab.research.google.com/github/Kashyapdhar05/KashyapDhar_Deep_Learning-/blob/main/exp_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# 1. Load Dataset (Assuming Alcohol_Sales.csv from FRED/Kaggle)
# Format: DATE (Index), Sales (Value)
df = pd.read_csv('Alcohol_Sales.csv', index_col='DATE', parse_dates=True)
df.columns = ['Sales']

# 2. Train/Test Split
# We'll use the last 12 months as testing data to see how well it forecasts a year
train = df.iloc[:len(df)-12]
test = df.iloc[len(df)-12:]

# 3. Scale Data
# RNNs are sensitive to the scale of input data (0-1 is ideal)
scaler = MinMaxScaler()
scaler.fit(train)
scaled_train = scaler.transform(train)
scaled_test = scaler.transform(test)

# 4. Create Time Series Generator
# n_input: How many past months the model looks at to predict the next month
n_input = 12
n_features = 1 # We are only predicting based on 'Sales'
generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)

# 5. Define LSTM Model
model = Sequential([
    LSTM(100, activation='relu', input_shape=(n_input, n_features), return_sequences=False),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(generator, epochs=50)

# 6. Forecasting (Rolling Window)
test_predictions = []
first_eval_batch = scaled_train[-n_input:]
current_batch = first_eval_batch.reshape((1, n_input, n_features))

for i in range(len(test)):
    # Predict one step ahead
    current_pred = model.predict(current_batch)[0]
    test_predictions.append(current_pred)

    # Update batch to include prediction and drop first value
    current_batch = np.append(current_batch[:,1:,:], [[current_pred]], axis=1)

# 7. Inverse Transform and Plot
true_predictions = scaler.inverse_transform(test_predictions)
test['Predictions'] = true_predictions
test.plot(figsize=(12,6))
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'Alcohol_Sales.csv'